<hr style="color:#808080;">
<p align="center"><b>N E P T U N E</b></p>
<hr style="color:#808080;">

In [5]:
# ---------- Libraries ----------
#
import torch

from neptune.data import DATASET_CONDITIONING, DATASET_VARIABLES_OCEAN, DATASET_VARIABLES_SURFACE, X, Y, Z
from neptune.data.weights import get_weights_mask, get_weights_mesh
from neptune.loss import loss_crps
from neptune.model import FGN
from neptune.model.fgn import _stack_inputs

# ---------- Jupyter ----------
#
%reload_ext autoreload
%autoreload 2

# ---------- Device ----------
#
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


<hr style="color:#808080;">
<p align="center"><b>T E N S O R S</b></p>
<hr style="color:#808080;">

In [6]:
# ---------- Tensors | Sizes of every input of the FGN, for a batch B and N previous states ----------
#
B, N = 1, 2
C_C  = 2 + len(DATASET_CONDITIONING)  # Surface conditioning, date (sin, cos) and ERA5 of x_t

mask_surface, mask_ocean = get_weights_mask()
mesh_surface, mesh_ocean = get_weights_mesh()

x_inp_s = torch.zeros(B, N, len(DATASET_VARIABLES_SURFACE), Y, X)
x_inp_o = torch.zeros(B, N, len(DATASET_VARIABLES_OCEAN), Z, Y, X)
cond_s  = torch.zeros(B, C_C, Y, X)
cond_o  = torch.zeros(B, 2, Z, Y, X)

# Encoder inputs | States, mask, mesh and conditioning stacked along the channels
x_s = _stack_inputs(x_inp_s, cond_s, mask_surface, mesh_surface)
x_o = _stack_inputs(x_inp_o, cond_o, mask_ocean, mesh_ocean)

for name, t in [
    ("Previous states (surface)", x_inp_s),
    ("Previous states (ocean)", x_inp_o),
    ("Conditioning (surface)", cond_s),
    ("Conditioning (ocean)", cond_o),
    ("Mask (surface)", mask_surface),
    ("Mask (ocean)", mask_ocean),
    ("Mesh (surface)", mesh_surface),
    ("Mesh (ocean)", mesh_ocean),
    ("Encoder inputs (surface)", x_s),
    ("Encoder inputs (ocean)", x_o),
]:
    print(f"{name:<26} | {str(tuple(t.shape)):<28} | {t.numel() * 4 / 1e9:.2f} GB (float32)")

Previous states (surface)  | (1, 2, 3, 256, 576)          | 0.00 GB (float32)
Previous states (ocean)    | (1, 2, 10, 48, 256, 576)     | 0.57 GB (float32)
Conditioning (surface)     | (1, 8, 256, 576)             | 0.00 GB (float32)
Conditioning (ocean)       | (1, 2, 48, 256, 576)         | 0.06 GB (float32)
Mask (surface)             | (1, 256, 576)                | 0.00 GB (float32)
Mask (ocean)               | (1, 48, 256, 576)            | 0.03 GB (float32)
Mesh (surface)             | (4, 256, 576)                | 0.00 GB (float32)
Mesh (ocean)               | (6, 48, 256, 576)            | 0.17 GB (float32)
Encoder inputs (surface)   | (1, 19, 256, 576)            | 0.01 GB (float32)
Encoder inputs (ocean)     | (1, 29, 48, 256, 576)        | 0.82 GB (float32)


<hr style="color:#808080;">
<p align="center"><b>C O N F I G U R A T I O N</b></p>
<hr style="color:#808080;">

In [7]:
# ---------- Configuration | Architecture of the FGN, as in experiments/forecast-ambient-increments/configs/train.yml ----------
#
config = {
    "input_states"          : 2,
    "cond_channels_surface" : 2 + len(DATASET_CONDITIONING),  # Added by train.py, date (sin, cos) and ERA5 of x_t
    "lat_channels"     : 256,
    "mod_features"     : 32,
    "config_surface"   : {
        "hid_channels"  : [32, 64, 128, 128, 256],
        "hid_blocks"    : [3, 3, 3, 3, 3],
        "hid_groups"    : [0, 0, 0, 0, 0],
        "kernel_size"   : 3,
        "stride"        : 2,
        "ffn_factor"    : 1,
        "dropout"       : 0.02,
        "checkpointing" : True,
    },
    "config_ocean"     : {
        "hid_channels"  : [32, 64, 128, 128, 256],
        "hid_blocks"    : [3, 3, 3, 3, 3],
        "hid_groups"    : [0, 0, 0, 0, 0],
        "kernel_size"   : 3,
        "stride"        : 2,
        "ffn_factor"    : 1,
        "dropout"       : 0.02,
        "checkpointing" : True,
    },
    "config_processor" : {
        "hid_channels"    : 1024,
        "hid_blocks"      : 16,
        "attention_heads" : 8,
        "ffn_factor"      : 2,
        "ffn_activation"  : "silu",
        "dropout"         : 0.02,
        "checkpointing"   : True,
    },
}

<hr style="color:#808080;">
<p align="center"><b>C O M P R E S S I O N</b></p>
<hr style="color:#808080;">

In [8]:
# ---------- Compression | Latent tokens of the configuration ----------
#
fgn = FGN(**config)
surface, ocean, total = fgn.compression()

parameters = sum(p.numel() for p in fgn.parameters())
print(f"Parameters          : {parameters / 1e6:.1f} M ({parameters * 4 / 1e9:.2f} GB in float32)")
print(f"Latent grids : surface {fgn.shape_surface} | ocean {fgn.shape_ocean}")
print(f"Tokens       : surface {fgn.tokens[0]} + ocean {fgn.tokens[1]} = {sum(fgn.tokens)}")
print(f"Compression  : surface ×{surface:.0f} | ocean ×{ocean:.0f} | total ×{total:.0f}")

Parameters          : 221.3 M (0.89 GB in float32)
Latent grids : surface [16, 36] | ocean [3, 16, 36]
Tokens       : surface 576 + ocean 1728 = 2304
Compression  : surface ×3 | ocean ×160 | total ×121


<hr style="color:#808080;">
<p align="center"><b>M E M O R Y</b></p>
<hr style="color:#808080;">

In [ ]:
# ---------- Memory | Forward pass (bf16) and gradient of a training step, for a batch B ----------
#
B, E = 1, 2  # Batch size per GPU and ensemble members

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

fgn = FGN(**config).to(DEVICE)
N   = config["input_states"]

x_inp_s = torch.randn(B, N, len(DATASET_VARIABLES_SURFACE), Y, X, device=DEVICE)
x_inp_o = torch.randn(B, N, len(DATASET_VARIABLES_OCEAN), Z, Y, X, device=DEVICE)
cond_s  = torch.randn(B, config["cond_channels_surface"], Y, X, device=DEVICE)  # Date and ERA5 of x_t
cond_o  = torch.randn(B, 2, Z, Y, X, device=DEVICE)  # Date of x_t
x_s     = torch.randn(B, len(DATASET_VARIABLES_SURFACE), Y, X, device=DEVICE)
x_o     = torch.randn(B, len(DATASET_VARIABLES_OCEAN), Z, Y, X, device=DEVICE)

with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
    x_pred_s, x_pred_o = fgn(x_inp_s, x_inp_o, cond_s, cond_o, members=E)

loss_s, loss_o = loss_crps(x_pred_s.float(), x_pred_o.float(), x_s, x_o, fgn.mask_surface, fgn.mask_ocean)
(loss_s + loss_o).backward()

parameters = sum(p.numel() for p in fgn.parameters())
print(f"Parameters          : {parameters / 1e6:.1f} M ({parameters * 4 / 1e9:.2f} GB in float32)")
print(f"Peak allocated [GB] : {torch.cuda.max_memory_allocated() / 1e9:.2f}")
print(f"Peak reserved  [GB] : {torch.cuda.max_memory_reserved() / 1e9:.2f}")
print(f"Not included [GB]   : {parameters * 8 / 1e9:.2f} (optimizer states, twice the parameters with AdamW, a bit more with SOAP)")

<hr style="color:#808080;">
<p align="center"><b>R O L L O U T</b></p>
<hr style="color:#808080;">

In [4]:
# ---------- Rollout | Autoregressive training step over ROLLOUT days (bf16): CRPS of every day and gradient ----------
#
import time

ROLLOUT = 3     # Number of autoregressive steps (days), 1 being the training step of the Memory section
B, E    = 1, 2  # Batch size per GPU and ensemble members

# Releasing the tensors of the previous sections
fgn = x_inp_s = x_inp_o = x_pred_s = x_pred_o = loss_s = loss_o = None
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

fgn = FGN(**config).to(DEVICE)
N   = config["input_states"]

# Each member is rolled out as a sample of its own (B * E), the targets being shared by the members (B)
x_inp_s = torch.randn(B * E, N, len(DATASET_VARIABLES_SURFACE), Y, X, device=DEVICE)
x_inp_o = torch.randn(B * E, N, len(DATASET_VARIABLES_OCEAN), Z, Y, X, device=DEVICE)
cond_s  = torch.randn(B * E, ROLLOUT, config["cond_channels_surface"], Y, X, device=DEVICE)  # Date and ERA5 of x_t, per step
cond_o  = torch.randn(B * E, ROLLOUT, 2, Z, Y, X, device=DEVICE)  # Date of x_t, per step
x_s     = torch.randn(B, ROLLOUT, len(DATASET_VARIABLES_SURFACE), Y, X, device=DEVICE)
x_o     = torch.randn(B, ROLLOUT, len(DATASET_VARIABLES_OCEAN), Z, Y, X, device=DEVICE)

torch.cuda.synchronize()
start = time.time()

loss = 0.0
for k in range(ROLLOUT):
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        x_pred_s, x_pred_o = fgn(x_inp_s, x_inp_o, cond_s[:, k], cond_o[:, k])

    # CRPS of the day, the members being gathered along the ensemble dimension, (B * E, ...) → (B, E, ...)
    loss_s, loss_o = loss_crps(
        x_pred_s[:, 0].unflatten(0, (B, E)).float(),
        x_pred_o[:, 0].unflatten(0, (B, E)).float(),
        x_s[:, k],
        x_o[:, k],
        fgn.mask_surface,
        fgn.mask_ocean,
    )
    loss = loss + (loss_s + loss_o) / ROLLOUT

    # Sliding window | The forecast of each member becomes its next input
    x_inp_s = torch.cat([x_inp_s[:, 1:], x_pred_s], dim=1)
    x_inp_o = torch.cat([x_inp_o[:, 1:], x_pred_o], dim=1)

loss.backward()
torch.cuda.synchronize()

parameters = sum(p.numel() for p in fgn.parameters())
print(f"Rollout             : {ROLLOUT} day(s), batch {B} x {E} members")
print(f"Time [s]            : {time.time() - start:.1f} (forward and backward)")
print(f"Peak allocated [GB] : {torch.cuda.max_memory_allocated() / 1e9:.2f}")
print(f"Peak reserved  [GB] : {torch.cuda.max_memory_reserved() / 1e9:.2f}")
print(f"Not included [GB]   : {parameters * 8 / 1e9:.2f} (optimizer states, twice the parameters with AdamW, a bit more with SOAP)")

Rollout             : 3 day(s), batch 1 x 2 members
Time [s]            : 6.0 (forward and backward)
Peak allocated [GB] : 76.34
Peak reserved  [GB] : 84.11
Not included [GB]   : 1.77 (optimizer states, twice the parameters with AdamW, a bit more with SOAP)
